# 🧹 Data Cleaning Pipeline - Telco Customer Churn

**Purpose:** Prepare raw Telco data for analysis and modelling.  
**Input:** `data/raw/telco.csv`  
**Output:** `data/processed/cleaned_telco.csv`

### Steps
1. [Load & inspect raw data](#1-load--inspect)
2. [Fix data types](#2-fix-data-types)
3. [Handle missing values](#3-handle-missing-values)
4. [Remove duplicates](#4-remove-duplicates)
5. [Standardise column names](#5-standardise-column-names)
6. [Engineer new features](#6-feature-engineering)
7. [Validate & save](#7-validate--save)

In [9]:
import pandas as pd
import numpy as np

RAW_PATH       = '../data/raw/telco.csv'
PROCESSED_PATH = '../data/processed/cleaned_telco.csv'

## 1. Load & Inspect

In [10]:
df = pd.read_csv(RAW_PATH)

print(f'Shape: {df.shape}')
print(f'\nDtypes:\n{df.dtypes}')
df.head()

Shape: (7043, 21)

Dtypes:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 2. Fix Data Types

`TotalCharges` is stored as `object` because some rows contain whitespace instead of a number.  
We coerce those to `NaN` and handle them in the next step.

In [11]:
# Fix BEFORE dropping nulls so coerced NaNs are also removed
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# SeniorCitizen is 0/1 int — convert to explicit boolean for clarity
df['SeniorCitizen'] = df['SeniorCitizen'].astype(bool)

print('TotalCharges dtype:', df['TotalCharges'].dtype)
print('NaNs introduced by coercion:', df['TotalCharges'].isna().sum())

TotalCharges dtype: float64
NaNs introduced by coercion: 11


## 3. Handle Missing Values

In [12]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({'missing_count': missing, 'missing_%': missing_pct})
print(missing_report[missing_report['missing_count'] > 0])

rows_before = len(df)
df = df.dropna()
print(f'\nDropped {rows_before - len(df)} rows with missing values. Remaining: {len(df)}')

              missing_count  missing_%
TotalCharges             11       0.16

Dropped 11 rows with missing values. Remaining: 7032


## 4. Remove Duplicates

In [13]:
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')
df = df.drop_duplicates()

# Sanity-check: customerID should be unique
id_dupes = df['customerID'].duplicated().sum()
print(f'Duplicate customerIDs after dedup: {id_dupes}')

Duplicate rows found: 0
Duplicate customerIDs after dedup: 0


## 5. Standardise Column Names

In [14]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print('Columns:', df.columns.tolist())

Columns: ['customerid', 'gender', 'seniorcitizen', 'partner', 'dependents', 'tenure', 'phoneservice', 'multiplelines', 'internetservice', 'onlinesecurity', 'onlinebackup', 'deviceprotection', 'techsupport', 'streamingtv', 'streamingmovies', 'contract', 'paperlessbilling', 'paymentmethod', 'monthlycharges', 'totalcharges', 'churn']


## 6. Feature Engineering

| Feature | Description |
|---|---|
| `avg_monthly_charge` | `totalcharges / tenure` — guarded against division by zero |
| `is_high_value` | `monthlycharges` above the median |
| `tenure_group` | Labelled tenure buckets for segmentation |
| `num_services` | Count of active add-on services per customer |
| `churn_binary` | Numeric encoding of the churn label |

In [15]:
# Guard against tenure=0 to avoid inf
df['avg_monthly_charge'] = np.where(
    df['tenure'] > 0,
    df['totalcharges'] / df['tenure'],
    df['monthlycharges']   # fallback: use current charge for brand-new customers
)

# High-value flag
df['is_high_value'] = df['monthlycharges'] > df['monthlycharges'].median()

# Tenure groups
df['tenure_group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12m', '13-24m', '25-48m', '49-72m'],
    right=True
)

# Count add-on services (Yes = active)
service_cols = [
    'onlinesecurity', 'onlinebackup', 'deviceprotection',
    'techsupport', 'streamingtv', 'streamingmovies'
]
df['num_services'] = df[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)

# Binary churn label
df['churn_binary'] = (df['churn'] == 'Yes').astype(int)

print('New columns added:')
df[['customerid', 'tenure', 'monthlycharges', 'totalcharges',
    'avg_monthly_charge', 'is_high_value', 'tenure_group',
    'num_services', 'churn_binary']].head()

New columns added:


,customerid,tenure,monthlycharges,totalcharges,avg_monthly_charge,is_high_value,tenure_group,num_services,churn_binary
0,7590-VHVEG,1,29.85,29.85,29.850000,False,0-12m,1,0
1,5575-GNVDE,34,56.95,1889.50,55.573529,False,25-48m,2,0
2,3668-QPYBK,2,53.85,108.15,54.075000,False,0-12m,2,1
3,7795-CFOCW,45,42.30,1840.75,40.905556,False,25-48m,3,0
4,9237-HQITU,2,70.70,151.65,75.825000,True,0-12m,0,1


## 7. Validate & Save

In [16]:
# --- Validation checks ---
assert df['customerid'].nunique() == len(df), 'customerID is not unique!'
assert df.isnull().sum().sum() == 0, 'Unexpected NaNs remain!'
assert (df['tenure'] >= 0).all(), 'Negative tenure values found!'
assert (df['monthlycharges'] > 0).all(), 'Non-positive monthlycharges found!'

print('✅ All validation checks passed.')
print(f'\nFinal dataset shape: {df.shape}')
print(f'Churn rate: {df["churn_binary"].mean():.1%}')

✅ All validation checks passed.

Final dataset shape: (7032, 26)
Churn rate: 26.6%


In [17]:
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved to {PROCESSED_PATH}')

Saved to ../data/processed/cleaned_telco.csv
